# 07 — V2 LLM features: simple supervisor experiment

**Goal:** test whether the redesigned V2 feature representation captures more useful signal
than the original V1 representation.

The notebook intentionally stays simple:

1. Load the 241 labelled training transcripts only.
2. Use the V2 label-blind prompt with `llm-feature-gen`.
3. Inspect a few outputs.
4. Convert the JSON into simple counts and per-100-word rates.
5. Compare Logistic Regression models using the same 5-fold cross-validation:
   transcript length, V1 LLM features (if available), V2 features, character TF-IDF.

**The test set is not loaded or used here.**

---

### Why the previous run died with `502 / 504 Bad Gateway`, and what changed

A 504 is the proxy in front of the model giving up while waiting. Four things made that
likely, and all four are fixed below — the rest of the notebook is unchanged.

| Cause | Fix |
|---|---|
| The request is not streamed, so the connection sits silent for a minute or more while a 122B model thinks. Idle connections are exactly what a gateway cuts. | **Stream the response.** Tokens arrive continuously, the connection is never idle, and the chunks are reassembled before parsing. |
| Qwen 3.5 reasons before answering, which lengthens every request and sometimes consumes the whole token budget so the visible answer comes back empty (`Invalid JSON response:` with nothing after the colon). | **`extra_body={"think": False}`**, passed from the notebook rather than patched into the library. |
| `LocalProvider` retries rate limits only. A single 504 propagates and ends the run. | **Retry with exponential backoff** on timeouts, 5xx, empty completions and unparseable JSON. |
| Raw responses are written once, at the very end. A failure at document 200 loses 200 completed calls. | **Append each response to disk as it arrives**, and skip documents already done. A rerun resumes; a rerun after success makes no calls at all. |

In [17]:
!pip install llm-feature-gen

## Configuration

In [18]:
from pathlib import Path
import zipfile

DATASET_FILENAME = "fileDataset.zip"

# Local setup (no Google Drive): the dataset archive ships with the repo at
# notebooks_preliminary/fileDataset.zip. Unpack it beside this notebook.
if not Path("fileDataset").exists():
    if not Path(DATASET_FILENAME).exists():
        raise FileNotFoundError(
            f"{DATASET_FILENAME} not found next to this notebook "
            "(it is committed at notebooks_preliminary/fileDataset.zip)."
        )
    with zipfile.ZipFile(DATASET_FILENAME) as zf:
        zf.extractall(".")
    print("unpacked", DATASET_FILENAME)
else:
    print("fileDataset/ already present")


Removing existing fileDataset.zip to ensure a fresh download.


Downloading...
From: https://drive.google.com/uc?id=1x6E98ldF-3-k6X3iwklEjNkm55x9e_Ck
To: /content/fileDataset.zip
100%|██████████| 191k/191k [00:00<00:00, 53.7MB/s]

Successfully downloaded fileDataset.zip.


### Important: Replace `YOUR_DATASET_DOWNLOAD_URL_HERE`

The code above includes a placeholder URL (`'YOUR_DATASET_DOWNLOAD_URL_HERE'`) for downloading `fileDataset.zip`. You **must** replace this with the actual URL where your dataset can be downloaded. If `fileDataset.zip` is already uploaded to your Colab environment, you can remove or comment out the code in the previous cell.

In [19]:
from pathlib import Path
import json
import re
import time
import zipfile
import numpy as np
import pandas as pd

RUN_LLM = True           # set False to re-use the cached responses in outputs/
MODEL = "qwen3.5:122b"   # use "qwen3.6:35b" for a faster preliminary run
BASE_URL = "https://llm.vse.cz/ollama/v1/"
API_KEY = "ollama"
MAX_TOKENS = 4096
TEMPERATURE = 0.0

# --- anti-504 settings --------------------------------------------------------
STREAM = True            # keeps the connection alive; the single most important fix
DISABLE_THINKING = True  # shorter requests, and no empty completions
REQUEST_TIMEOUT = 900    # seconds, per request
MAX_RETRIES = 5
LIMIT_DOCS = None        # set to e.g. 20 for a trial run; None = all 241


def find_dataset(start=Path(".")):
    """Locate fileDataset/, unzipping fileDataset.zip if only the zip is present."""
    ok = lambda p: (p / "train").is_dir()
    for c in [start / "fileDataset", start / "../fileDataset", start,
              start / "data" / "fileDataset", start / "fileDataset_data",
              start / "fileDataset" / "fileDataset"]:
        if c.is_dir() and ok(c):
            return c.resolve()
    zips = sorted(start.rglob("fileDataset.zip")) + sorted(start.parent.glob("fileDataset.zip"))
    if not zips:
        raise FileNotFoundError(
            "Could not find fileDataset/train. Put fileDataset.zip or the unpacked folder "
            "next to this notebook.")
    target = (start / "fileDataset_data").resolve()
    target.mkdir(exist_ok=True)
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall(target)
    print("unpacked", zips[0].name, "->", target)
    for c in [target, target / "fileDataset"]:
        if ok(c):
            return c.resolve()
    raise FileNotFoundError(f"nothing usable inside {target}")


DATA = find_dataset()
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)

print("Data:", DATA)
print("Model:", MODEL, "| stream:", STREAM, "| thinking disabled:", DISABLE_THINKING)
print("RUN_LLM:", RUN_LLM)

unpacked fileDataset.zip -> /content/fileDataset_data
Data: /content/fileDataset_data
Model: qwen3.5:122b | stream: True | thinking disabled: True
RUN_LLM: True


## 1. Load the training data

Labels come from the directory names: `train/negative` → 0, `train/positive` → 1.
Only these 241 labelled documents are used; `test/` is never read.

In [20]:
rows = []

for label_name, label in [("negative", 0), ("positive", 1)]:
    for path in sorted((DATA / "train" / label_name).glob("*.txt")):
        text = path.read_text(encoding="utf-8").strip()
        rows.append({
            "file": path.name,
            "label": label,
            "text": text,
            "n_word": len(text.split()),
        })

train = pd.DataFrame(rows)
if LIMIT_DOCS:
    train = (train.groupby("label", group_keys=False)
             .apply(lambda g: g.head(max(1, round(LIMIT_DOCS * len(g) / len(rows)))))
             .reset_index(drop=True))

print(f"{len(train)} transcripts")
print(train["label"].value_counts().sort_index())
print(f"Median words: {train.n_word.median():.0f}")
train.head(3)

241 transcripts
label
0    171
1     70
Name: count, dtype: int64
Median words: 70


,file,label,text,n_word
0,100tr0.txt,0,Žena na lehátku čte knihu. Děti si házejí s mí...,66
1,101tr0.txt,0,"Takže muž plave. Dítě leží na břehu, má namoče...",69
2,102tr0.txt,0,"Muž plave ve vodě, za ním plave ryba. Za rybou...",98


## 2. V2 prompt

V2 asks the LLM to count observable linguistic events and return Czech evidence spans. The
important change from V1 is the representation: **counts rather than 3–5 level guesses**.

In [21]:
V2_PROMPT = r"""
You are annotating a transcript of spontaneous Czech speech. A person was shown a drawing of a
lakeshore scene and asked to describe it aloud. The text is an automatic transcription.

Your job is to COUNT observable linguistic events and QUOTE the evidence for each count.
You are an annotator, not an evaluator.

RULES
- Quote evidence verbatim from the transcript, in Czech. Never translate or paraphrase.
- If a category has no instances, return an empty list. Empty is a valid answer.
- Never infer anything about the speaker: not their health, ability, intelligence, age,
  education or state of mind. Describe the language, never the person.
- Do not compare this speaker to anyone else or to any norm.
- Count occurrences, not impressions. Two hedges in one sentence are two entries.
- Return only JSON.

CATEGORIES
1. named_entities - distinct objects, creatures or people explicitly named. Lemmatise and list
   each distinct entity exactly once (a set, not a list of mentions).
2. specific_action_verbs - verbs naming a particular manner of action. List every occurrence.
3. generic_verbs - verbs of bare existence, possession, location or unspecified movement.
4. complete_propositions - integer count of clauses with an explicit subject and predicate
   plus at least one further argument or adjunct.
5. locative_expressions - phrases placing something somewhere. List every occurrence.
6. regions_referenced - distinct regions among "water", "land", "sky".
7. hedge_spans - expressions of uncertainty. List every occurrence.
8. deictic_spans - references such as "there", "that thing" that substitute for naming.
9. metacomment_spans - remarks about the speaker's own describing, remembering, or the task.
10. repeated_content_lemmas - content words used more than once, with counts.
11. self_corrections - integer count.
12. diminutive_or_affective_forms - diminutive or affectionate noun forms.
13. quantity_expressions - numerals or quantifiers applied to things in the scene.

Return exactly this JSON and nothing else:
{
  "named_entities": [],
  "specific_action_verbs": [],
  "generic_verbs": [],
  "complete_propositions": 0,
  "locative_expressions": [],
  "regions_referenced": [],
  "hedge_spans": [],
  "deictic_spans": [],
  "metacomment_spans": [],
  "repeated_content_lemmas": [{"lemma": "", "count": 0}],
  "self_corrections": 0,
  "diminutive_or_affective_forms": [],
  "quantity_expressions": []
}
"""

# Simple label/domain leakage check
bad = ["two hidden", "two categories", "two classes", "diagnos", "dementia",
       "alzheim", "cognitive decline", "control group", "healthy"]
found = [x for x in bad if x in V2_PROMPT.lower()]
print("Label/domain leak check:", "PASS" if not found else found)

Label/domain leak check: PASS


## 3. The provider — this is the 504 fix

A small subclass of the library's `LocalProvider`, overriding the one method every call goes
through. Streaming, no-think, and retries. Nothing in the installed library is modified, so
whatever state your local copy is in, this notebook behaves the same.

In [22]:
from openai import BadRequestError
from llm_feature_gen.providers.local_provider import LocalProvider

THINK_TAGS = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)


class StreamingLocalProvider(LocalProvider):
    """LocalProvider + streaming + no-think + retries. Same public interface."""

    def __init__(self, *args, stream=STREAM, think=not DISABLE_THINKING,
                 request_timeout=REQUEST_TIMEOUT, **kwargs):
        super().__init__(*args, **kwargs)
        self.stream = stream
        self.extra_body = {} if think else {"think": False}
        self.request_timeout = request_timeout
        self.n_calls = 0

    def _complete(self, model, system_prompt, user_content, kwargs):
        messages = [{"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_content}]
        common = dict(model=model, messages=messages, temperature=self.temperature,
                      max_tokens=self.max_tokens, timeout=self.request_timeout, **kwargs)
        if not self.stream:
            return self.client.chat.completions.create(**common).choices[0].message.content or ""
        chunks = []
        for event in self.client.chat.completions.create(stream=True, **common):
            if event.choices:
                piece = getattr(event.choices[0].delta, "content", None)
                if piece:
                    chunks.append(piece)
        return "".join(chunks)

    def _chat_json(self, deployment_name, system_prompt, user_content, json_mode=False):
        if json_mode and "JSON" not in system_prompt:
            system_prompt = system_prompt + " Respond in strict JSON format."

        kwargs = {}
        if json_mode:
            kwargs["response_format"] = {"type": "json_object"}
        if self.extra_body:
            kwargs["extra_body"] = dict(self.extra_body)

        backoff, last = 2, None
        for attempt in range(self.max_retries):
            try:
                self.n_calls += 1
                text = THINK_TAGS.sub(
                    "", self._complete(deployment_name, system_prompt, user_content, kwargs)).strip()
                if not text:
                    raise ValueError("empty completion (model returned no visible content)")
                try:
                    parsed = json.loads(text)
                except Exception:
                    parsed = self._extract_json(text)
                if not isinstance(parsed, dict):
                    raise ValueError(f"unparseable response: {text[:150]}")
                return parsed

            except BadRequestError as e:
                msg = str(e)
                if json_mode and "json_object" in msg:          # model has no JSON mode
                    json_mode = False
                    kwargs.pop("response_format", None)
                    continue
                if "extra_body" in kwargs and ("think" in msg.lower() or "unknown" in msg.lower()):
                    kwargs.pop("extra_body")                    # endpoint rejects the flag
                    print("    note: endpoint rejected {'think': False}; continuing without it")
                    continue
                last = e
            except Exception as e:
                last = e

            if attempt < self.max_retries - 1:
                print(f"    retry {attempt + 1}/{self.max_retries - 1} — "
                      f"{type(last).__name__}: {str(last)[:100]}")
                time.sleep(backoff)
                backoff = min(backoff * 2, 60)

        raise RuntimeError(f"gave up after {self.max_retries} attempts: {last}")


provider = None
if RUN_LLM:
    provider = StreamingLocalProvider(
        base_url=BASE_URL,
        api_key=API_KEY,
        default_text_model=MODEL,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        max_retries=MAX_RETRIES,
    )
    print("Provider ready:", provider.text_model,
          "| stream:", provider.stream, "| extra_body:", provider.extra_body)

Provider ready: qwen3.5:122b | stream: True | extra_body: {'think': False}


## Smoke test — run this first

One transcript. If this 504s, so will 241 of them, and you will know in a minute rather than
an hour. If it succeeds but takes more than ~60 s, switch `MODEL` to `qwen3.6:35b`.

In [23]:
if RUN_LLM:
    t0 = time.time()
    test_result = provider.text_features([train.iloc[0]["text"]], prompt=V2_PROMPT)[0]
    print(f"Elapsed: {time.time() - t0:.1f}s")
    print(json.dumps(test_result, ensure_ascii=False, indent=2)[:1500])
else:
    print("RUN_LLM=False — smoke test skipped.")

    retry 1/4 — PermissionDeniedError: <html>
<head><title>403 Forbidden</title></head>
<body>
<center><h1>403 Forbidden</h1></center>

    retry 2/4 — PermissionDeniedError: <html>
<head><title>403 Forbidden</title></head>
<body>
<center><h1>403 Forbidden</h1></center>

    retry 3/4 — PermissionDeniedError: <html>
<head><title>403 Forbidden</title></head>
<body>
<center><h1>403 Forbidden</h1></center>

    retry 4/4 — PermissionDeniedError: <html>
<head><title>403 Forbidden</title></head>
<body>
<center><h1>403 Forbidden</h1></center>



RuntimeError: gave up after 5 attempts: <html>
<head><title>403 Forbidden</title></head>
<body>
<center><h1>403 Forbidden</h1></center>
<hr><center>nginx</center>
</body>
</html>

## Full extraction — resumable

Each response is appended to `outputs/v2_simple_raw.jsonl` the moment it arrives and flushed
to disk. If the run dies at document 147, rerun this cell and it continues from 147.

In [ ]:
RAW_PATH = OUT / "v2_simple_raw.jsonl"


def load_raw(path=RAW_PATH):
    if not path.exists():
        return {}
    out = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rec = json.loads(line)
            out[rec["file"]] = rec
    return out


raw_records = load_raw()

if RUN_LLM:
    todo = train[~train.file.isin(raw_records)]
    print(f"{len(raw_records)} already done, {len(todo)} to go")
    t0 = time.time()
    with RAW_PATH.open("a", encoding="utf-8") as fh:
        for i, row in enumerate(todo.itertuples(), 1):
            try:
                rec = {"file": row.file, "ok": True,
                       "response": provider.text_features([row.text], prompt=V2_PROMPT)[0]}
            except Exception as e:
                rec = {"file": row.file, "ok": False, "error": str(e)[:300], "response": {}}
                print(f"  FAILED {row.file}: {str(e)[:110]}")
            fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
            fh.flush()                                    # survive a kernel death
            raw_records[row.file] = rec
            if i % 10 == 0 or i == len(todo):
                per = (time.time() - t0) / i
                print(f"  {i}/{len(todo)} | {per:.1f}s/doc | "
                      f"~{per * (len(todo) - i) / 60:.0f} min left")
    print(f"\nDone in {(time.time() - t0) / 60:.1f} min | {provider.n_calls} HTTP calls")
elif not raw_records:
    raise FileNotFoundError(f"{RAW_PATH} not found. Run with RUN_LLM=True first.")

# Keep the plain {file: response} shape the rest of the notebook expects.
raw = {f: r["response"] for f, r in raw_records.items() if r.get("ok")}
n_failed = sum(1 for r in raw_records.values() if not r.get("ok"))
print(f"{len(raw)} usable responses, {n_failed} failed "
      f"({n_failed / max(len(raw_records), 1):.1%})")

# Only model documents we actually have a response for.
train = train[train.file.isin(raw)].reset_index(drop=True)
print(f"modelling on {len(train)} documents ({int(train.label.sum())} positive)")

## 4. Inspect the extracted features

Before modelling, inspect several responses. We mainly want to know: are the counts plausible,
are evidence spans actually present in the transcript, and is the model inventing evidence?

In [ ]:
for file_name in list(raw)[:3]:
    print("\nFILE:", file_name)
    print(json.dumps(raw[file_name], ensure_ascii=False, indent=2)[:2500])

In [ ]:
# Simple groundedness check: evidence strings should occur in the transcript

EVIDENCE_FIELDS = [
    "specific_action_verbs", "locative_expressions", "hedge_spans",
    "deictic_spans", "metacomment_spans",
    "diminutive_or_affective_forms", "quantity_expressions"
]


def groundedness(row, result):
    checks = []
    for field in EVIDENCE_FIELDS:
        for span in result.get(field, []) or []:
            if isinstance(span, str):
                checks.append(span.lower().strip() in row["text"].lower())
    return np.mean(checks) if checks else np.nan


grounded = [groundedness(row, raw.get(row["file"], {})) for _, row in train.iterrows()]
grounded = pd.Series(grounded)

print(f"Grounded evidence rate : {grounded.mean():.1%}")
print(f"Documents with any invented span: {(grounded < 1).mean():.1%}")
print(f"Documents with no spans at all  : {grounded.isna().mean():.1%}")

A span the model quoted that is not in the transcript means the count that produced it is
fabricated. If this rate is poor, no model result below can be trusted — and that is itself a
reportable finding about extraction reliability.

## 5. Convert V2 into simple ML variables

Counts are kept, but we also calculate **rates per 100 words** so that a longer transcript does
not automatically receive larger counts. This is one of the main changes from V1.

In [ ]:
COUNT_MAP = {
    "named_entity_count": "named_entities",
    "specific_action_count": "specific_action_verbs",
    "generic_verb_count": "generic_verbs",
    "locative_count": "locative_expressions",
    "hedge_count": "hedge_spans",
    "deictic_count": "deictic_spans",
    "metacomment_count": "metacomment_spans",
    "diminutive_count": "diminutive_or_affective_forms",
    "quantity_count": "quantity_expressions",
}


def to_features(row, result):
    x = {"n_word": row["n_word"]}

    for out_name, field in COUNT_MAP.items():
        value = result.get(field, [])
        x[out_name] = len(value) if isinstance(value, list) else float(value or 0)
        x[out_name + "_rate"] = x[out_name] / max(row["n_word"], 1) * 100

    x["complete_propositions"] = float(result.get("complete_propositions", 0) or 0)
    x["self_corrections"] = float(result.get("self_corrections", 0) or 0)

    reps = result.get("repeated_content_lemmas", []) or []
    x["repeated_lemma_count"] = len([
        r for r in reps
        if isinstance(r, dict) and (r.get("count") or 0) > 1
    ])

    regions = result.get("regions_referenced", []) or []
    x["regions_count"] = len(regions) if isinstance(regions, list) else 0

    if x["specific_action_count"] + x["generic_verb_count"] > 0:
        x["specific_verb_ratio"] = (
            x["specific_action_count"] /
            (x["specific_action_count"] + x["generic_verb_count"])
        )
    else:
        x["specific_verb_ratio"] = 0.0

    return x


X_v2 = pd.DataFrame(
    [to_features(row, raw.get(row["file"], {})) for _, row in train.iterrows()],
    index=train.index,
)

print(X_v2.shape)
X_v2.head()

In [ ]:
# Sanity: do the rates still track transcript length?
COUNTS = [c for c in X_v2.columns if not c.endswith("_rate") and c != "n_word"]
r2 = pd.DataFrame([{
    "feature": c,
    "r2_count": np.corrcoef(X_v2[c], X_v2.n_word)[0, 1] ** 2,
    "r2_rate": (np.corrcoef(X_v2[c + "_rate"], X_v2.n_word)[0, 1] ** 2
                if c + "_rate" in X_v2 else np.nan),
} for c in COUNTS if X_v2[c].std() > 0])
print(r2.sort_values("r2_count", ascending=False).to_string(
    index=False, float_format=lambda v: f"{v:.3f}"))

## 6. Simple model comparison

One model family is enough for this preliminary experiment: **Logistic Regression**.
Every representation uses the same stratified 5-fold CV.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, balanced_accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer

y = train["label"].values
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
logreg = lambda: LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)


def evaluate(name, X, model, target=None):
    yy = y if target is None else target
    p = cross_val_predict(model, X, yy, cv=cv, method="predict_proba")[:, 1]
    pred = (p >= 0.5).astype(int)
    return {
        "representation": name,
        "AUC": roc_auc_score(yy, p),
        "F1": f1_score(yy, pred),
        "Accuracy": accuracy_score(yy, pred),
        "Balanced Accuracy": balanced_accuracy_score(yy, pred),
    }


results = [
    evaluate("Transcript length", X_v2[["n_word"]],
             make_pipeline(StandardScaler(), logreg())),
]

# V2: use the normalized rates + scale-free ratio
V2_COLS = [c for c in X_v2.columns if c.endswith("_rate")] + [
    "specific_verb_ratio", "regions_count", "complete_propositions",
    "self_corrections", "repeated_lemma_count",
]
V2_COLS = [c for c in V2_COLS if c in X_v2.columns]

results.append(evaluate("V2 LLM features", X_v2[V2_COLS],
                        make_pipeline(StandardScaler(), logreg())))

results.append(evaluate("TF-IDF char 3-5gram", train["text"],
                        make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                                                      min_df=3, sublinear_tf=True), logreg())))

results_df = pd.DataFrame(results).sort_values("AUC", ascending=False)
results_df

## Optional: add the existing V1 LLM features

If `OutputsQwen/train_all_feature_values.csv` exists, this adds V1 to the same comparison.
V1 is evaluated on exactly the documents both representations cover, so the two numbers are
comparable.

In [ ]:
V1_CANDIDATES = [
    Path("OutputsQwen/train_all_feature_values.csv"),
    Path("outputs/features_v1_train.csv"),
    Path("train_all_feature_values.csv"),
]
V1_PATH = next((p for p in V1_CANDIDATES if p.exists()), None)

if V1_PATH is not None:
    v1 = pd.read_csv(V1_PATH)
    file_col = "File" if "File" in v1.columns else "file"
    v1_cols = [c for c in v1.columns
               if c not in {file_col, "Class", "class", "raw_llm_output"}]

    train_v1 = train.merge(v1, left_on="file", right_on=file_col,
                           how="inner", validate="1:1")
    print(f"V1 loaded: {V1_PATH} — covers {len(train_v1)}/{len(train)} documents")

    if len(train_v1) < len(train):
        print("  note: V1 and V2 cover different document sets; the AUCs below are still")
        print("  computed on their own rows, so treat the comparison as approximate.")

    p = cross_val_predict(
        make_pipeline(OneHotEncoder(handle_unknown="ignore", min_frequency=5), logreg()),
        train_v1[v1_cols], train_v1["label"], cv=cv, method="predict_proba")[:, 1]
    pred = (p >= 0.5).astype(int)

    results_df = pd.concat([results_df, pd.DataFrame([{
        "representation": "V1 LLM features",
        "AUC": roc_auc_score(train_v1["label"], p),
        "F1": f1_score(train_v1["label"], pred),
        "Accuracy": accuracy_score(train_v1["label"], pred),
        "Balanced Accuracy": balanced_accuracy_score(train_v1["label"], pred),
    }])], ignore_index=True).sort_values("AUC", ascending=False)
else:
    print("V1 CSV not found; skipping V1 comparison.")

results_df = results_df.reset_index(drop=True)
results_df.to_csv(OUT / "v2_simple_results.csv", index=False)
results_df

In [ ]:
# Record what produced these numbers.
import hashlib
from datetime import datetime, timezone

(OUT / "v2_simple_run_metadata.json").write_text(json.dumps({
    "model": MODEL, "base_url": BASE_URL, "temperature": TEMPERATURE,
    "max_tokens": MAX_TOKENS, "streaming": STREAM,
    "thinking_disabled": DISABLE_THINKING,
    "prompt_sha256": hashlib.sha256(V2_PROMPT.encode()).hexdigest()[:16],
    "n_documents": len(train), "n_failed": n_failed,
    "grounded_evidence_rate": round(float(grounded.mean()), 4),
    "run_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "split": "train only — test set not read",
}, indent=2), encoding="utf-8")
print((OUT / "v2_simple_run_metadata.json").read_text())

## 7. Takeaway

The number to focus on is **V2 LLM features vs V1 and transcript length**, using the same
cross-validation.

For the supervisor meeting, phrase the conclusion according to the result rather than deciding
it in advance:

- **V2 > V1:** the redesigned representation appears to recover signal lost by categorical encoding.
- **V2 ≈ V1:** the encoding may not have been the main bottleneck; extraction reliability becomes
  the next question.
- **V2 < V1:** inspect the quality and groundedness of the extracted counts before drawing
  conclusions.

TF-IDF remains the strong Czech lexical baseline, not something V2 is required to beat in this
preliminary experiment.

In [ ]:
def takeaway(df):
    get = lambda n: (float(df.loc[df.representation == n, "AUC"].iloc[0])
                     if (df.representation == n).any() else None)
    v2, v1, length = get("V2 LLM features"), get("V1 LLM features"), get("Transcript length")
    print(f"V2 LLM features   AUC {v2:.3f}")
    print(f"Transcript length AUC {length:.3f}")
    if v1 is not None:
        print(f"V1 LLM features   AUC {v1:.3f}")
    print()
    if v1 is not None and v2 > v1 + 0.05:
        print("V2 > V1 — the redesign recovers signal the categorical encoding was discarding.")
    elif v1 is not None and abs(v2 - v1) <= 0.05:
        print("V2 = V1 — the encoding was not the bottleneck. Check the groundedness rate and")
        print("the failure rate in section 4 before concluding anything about the constructs.")
    elif v1 is not None:
        print("V2 < V1 — most likely inconsistent counts rather than wrong constructs.")
        print("Section 4's groundedness rate is where to look.")
    if v2 <= length + 0.02:
        print("\nNote: V2 does not clearly beat counting the words in the transcript, so it is")
        print("not yet measuring language rather than sheer volume.")


takeaway(results_df)